# 04 — AMX Tech Churn Modeling
Phase 7 reviews the leakage-safe snapshot, stratified model comparison, held-out evaluation, errors, and model interpretation. Set `RETRAIN = True` to reproduce the complete experiment from PostgreSQL; the default loads the persisted report so review is fast.

In [ ]:
import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
RETRAIN = False

if RETRAIN:
    from sentinel.config import get_settings
    from sentinel.database.connection import create_database_engine
    from sentinel.database.loader import read_database_tables
    from sentinel.eda.cleaning import clean_dataset
    from sentinel.ml.training import train_churn_model

    engine = create_database_engine(get_settings().database_url)
    try:
        tables, _ = clean_dataset(read_database_tables(engine))
    finally:
        engine.dispose()
    _, report, ranking = train_churn_model(tables)
else:
    report = json.loads((PROJECT_ROOT / 'models/churn_model_report.json').read_text(encoding='utf-8'))
    ranking = pd.read_csv(PROJECT_ROOT / 'models/churn_test_predictions.csv')

report['company'], report['snapshot'], report['cohort']

## Evaluation protocol
The held-out test remains untouched during candidate selection, randomized tuning, and threshold selection. Average precision is primary because churn prevalence is low; accuracy is not optimized alone.

In [ ]:
cv = pd.DataFrame({
    name: {metric: values['mean'] for metric, values in metrics.items()}
    for name, metrics in report['cross_validation'].items()
}).T
cv.sort_values('average_precision', ascending=False).round(3)

In [ ]:
metrics = report['held_out_test_metrics']
pd.Series({key: value for key, value in metrics.items() if key != 'confusion_matrix'}, name='held_out').round(3)

In [ ]:
cm = metrics['confusion_matrix']
pd.DataFrame(
    [[cm['true_negative'], cm['false_positive']], [cm['false_negative'], cm['true_positive']]],
    index=['actual_not_churned', 'actual_churned'],
    columns=['predicted_not_churned', 'predicted_churned'],
)

In [ ]:
pd.DataFrame(report['permutation_importance']).head(12).round(4)

In [ ]:
ranking.head(15)

## Interpretation boundary
This synthetic held-out result demonstrates predictive ranking, not causation or guaranteed real-world performance. False negatives are missed retention opportunities; false positives consume outreach capacity. The threshold must eventually be validated against AMX Tech's actual intervention costs, calibration requirements, drift, and governance rules.